In [2]:
import os

import numpy as np
import pandas as pd

In [ ]:
zstr = ['0.0004', '0.002', '0.006', '0.014', '0.020']
logt_out = np.arange(6., 10.25, 0.05)

for i in range(len(zstr)):
    with open(os.path.join('Geneva_rot', 'original', f'isochrones_z{zstr[i]}.dat'), 'r') as f:
        lines = f.readlines()
        header, lines = lines[0], lines[1:]
        header = [h for h in header.strip().split(' ') if h != '']
        next_logt = 0.
        isoc_data = np.zeros((0, len(header)+1), dtype=float)
        for line in lines:
            data = [val for val in line.strip().split(' ') if val != '']  # handle multiple whitespaces between values
            # handle empty lines
            if len(data) == 0:
                continue
            # handle the header for each isochrone
            if data[0] == 'Isochrone':
                next_logt = float(data[4])
                continue
            # handle the actual data
            data = np.array([float(d) for d in data])
            data = np.concatenate(([next_logt], data))
            isoc_data = np.concatenate((isoc_data, data.reshape((1, len(data)))), axis=0)
        header = np.concatenate((['time'], header))

    print(header)
    print(len(header))
    print(isoc_data)
    print(isoc_data.shape)

    # Get the data that is needed for FSPS
    header_out = ['log(age)', 'Mini', 'Mact', 'logl', 'logt', 'logg', 'Composition', 'Phase']
    isoc_out = np.zeros((isoc_data.shape[0], 8))
    isoc_out[:,0] = isoc_data[:,0]  # log(age)
    isoc_out[:,1] = isoc_data[:,1]  # Mini
    isoc_out[:,2] = isoc_data[:,4]  # Mact
    isoc_out[:,3] = isoc_data[:,5]  # logl
    isoc_out[:,4] = isoc_data[:,6]  # logt
    isoc_out[:,5] = isoc_data[:,24] # logg
    # leave composition and phase as zeros
    logt = np.unique(isoc_out[:,0])
    nt = len(logt)

    # flag wolf-rayet stars: stars with H1 mass fractions < 0.4, log(Teff) > 4.4
    wr = np.where((isoc_data[:,33] < 0.4) & (isoc_data[:,6] > 4.4))
    isoc_out[wr,7] = 9.0

    # # pad with zeros 
    # ltlo = logt_out[logt_out < logt.min()]
    # lthi = logt_out[logt_out > logt.max()]
    # for ltl in ltlo:
    #     row = np.zeros((1,isoc_out.shape[1]))
    #     row[0,0] = ltl
    #     isoc_out = np.concatenate((row, isoc_out), axis=0)
    # for lth in lthi:
    #     row = np.zeros((1,isoc_out.shape[1]))
    #     row[0,0] = lth
    #     isoc_out = np.concatenate((isoc_out, row), axis=0)

    # print(isoc_out)
    # logt = np.unique(isoc_out[:,0])
    # nt = len(logt)

    # write it out in a format that FSPS likes
    header_out = '# ' + ' '.join(header_out)
    with open(os.path.join('Geneva_rot', f'isoc_z{float(zstr[i]):.4f}.dat'), 'w') as f:
        for logti in logt_out:
            f.write(header_out + '\n')
            wh = np.where(np.isclose(isoc_out[:,0], logti))[0]
            for row in wh:
                f.write('%8.2f%14.8f%9.4f%9.4f%9.4f%9.4f%9.4f%9.4f' % tuple(isoc_out[row,:]) + '\n')

    # np.savetxt(os.path.join('Geneva_rot', f'isoc_z{float(zstr[i]):.4f}.dat'), isoc_out, fmt='%8.2f%14.8f%9.4f%9.4f%9.4f%9.4f%9.4f%9.4f', header=header_out)

['time' 'M_ini' 'Z_ini' 'OmOc_ini' 'M' 'logL' 'logTe_c' 'logTe_nc' 'MBol'
 'MV' 'U-B' 'B-V' 'V-R' 'V-I' 'J-K' 'H-K' 'V-K' 'G-V' 'Gbp-V' 'Grp-V'
 'GFlag' 'BC' 'r_pol' 'oblat' 'g_pol' 'Omega_S' 'v_eq' 'v_crit1' 'v_crit2'
 'Om/Om_cr' 'lg(Md)' 'lg(Md_M)' 'Ga_Ed' 'H1' 'He4' 'C12' 'C13' 'N14' 'O16'
 'O17' 'O18' 'Ne20' 'Ne22' 'Al26']
44
[[6.000e+00 1.700e+00 4.000e-04 ... 5.363e-05 4.337e-06 0.000e+00]
 [6.000e+00 1.851e+00 4.000e-04 ... 5.363e-05 4.337e-06 0.000e+00]
 [6.000e+00 2.016e+00 4.000e-04 ... 5.363e-05 4.337e-06 0.000e+00]
 ...
 [9.100e+00 1.735e+00 4.000e-04 ... 5.363e-05 4.337e-06 0.000e+00]
 [9.100e+00 1.736e+00 4.000e-04 ... 5.363e-05 4.337e-06 0.000e+00]
 [9.100e+00 1.737e+00 4.000e-04 ... 5.363e-05 4.337e-06 0.000e+00]]
(5934, 44)
['time' 'M_ini' 'Z_ini' 'OmOc_ini' 'M' 'logL' 'logTe_c' 'logTe_nc' 'MBol'
 'MV' 'U-B' 'B-V' 'V-R' 'V-I' 'J-K' 'H-K' 'V-K' 'G-V' 'Gbp-V' 'Grp-V'
 'GFlag' 'BC' 'r_pol' 'oblat' 'g_pol' 'Omega_S' 'v_eq' 'v_crit1' 'v_crit2'
 'Om/Om_cr' 'lg(Md)' 'lg(Md_M)